## Local Inference on GPU
Model page: https://huggingface.co/google/medgemma-1.5-4b-it

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/google/medgemma-1.5-4b-it)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

The model you are trying to use is gated. Please make sure you have access to it by visiting the model page.To run inference, either set HF_TOKEN in your environment variables/ Secrets or run the following cell to login. 🤗

In [1]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print("✅ Ready")


✅ Ready


In [2]:
import os, torch
from transformers import pipeline
from huggingface_hub import get_token

hf_token = get_token() or os.environ.get("HF_TOKEN")
if not hf_token:
    from huggingface_hub import login
    login()
    hf_token = get_token()

pipe = pipeline(
    "image-text-to-text",
    model="google/medgemma-1.5-4b-it",
    token=hf_token,
    device_map="auto",
    dtype=torch.bfloat16,
)
print("✅ MedGemma loaded")


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

✅ MedGemma loaded


In [3]:
import os, re, threading, time, subprocess, base64, io
from flask import Flask, request, jsonify
from PIL import Image
import torch

os.system("fuser -k 8000/tcp 2>/dev/null; true")
time.sleep(1)

app = Flask(__name__)

def clean_response(text: str) -> str:
    """Strip MedGemma thinking tokens, keep only the final answer."""
    # <think>...</think> XML-style
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    # <unusedN>thought\n...blank-line token-style
    text = re.sub(r'<unused\d+>thought\n.*?\n\n', '', text, flags=re.DOTALL)
    # Any remaining bare <unusedN> tokens
    text = re.sub(r'<unused\d+>[^\n]*\n?', '', text)
    return text.strip()

@app.route("/v1/chat/completions", methods=["POST"])
def chat():
    try:
        body        = request.get_json()
        messages    = body.get("messages", [])
        # Hard-cap at 500 — MedGemma thinks first (~300 tokens) then answers,
        # so 500 visible tokens already costs ~800 total. Keep T4 from OOMing.
        max_tokens  = min(int(body.get("max_tokens", 400)), 500)

        hf_messages = []
        for m in messages:
            raw_content = m["content"]
            if isinstance(raw_content, str):
                raw_content = [{"type": "text", "text": raw_content}]

            hf_content = []
            for part in raw_content:
                if part.get("type") == "image_url":
                    url = part["image_url"]["url"]
                    if url.startswith("data:"):
                        _, data = url.split(",", 1)
                        img = Image.open(io.BytesIO(base64.b64decode(data)))
                        hf_content.append({"type": "image", "image": img})
                    else:
                        hf_content.append({"type": "image", "url": url})
                else:
                    hf_content.append(part)

            hf_messages.append({"role": m["role"], "content": hf_content})

        try:
            output = pipe(hf_messages, max_new_tokens=max_tokens, do_sample=False)
        finally:
            torch.cuda.empty_cache()   # free KV-cache after every call

        generated = output[0]["generated_text"]
        last = generated[-1]["content"] if isinstance(generated, list) else generated
        if isinstance(last, list):
            last = " ".join(p.get("text", "") for p in last if p.get("type") == "text")

        last = clean_response(last)

        return jsonify({
            "choices": [{"message": {"role": "assistant", "content": last}}]
        })

    except Exception as e:
        import traceback
        err = traceback.format_exc()
        print(f"\n❌ ERROR:\n{err}")
        torch.cuda.empty_cache()
        return jsonify({"error": str(e), "trace": err}), 500


threading.Thread(
    target=lambda: app.run(host="0.0.0.0", port=8000, use_reloader=False),
    daemon=True,
).start()
time.sleep(2)

def start_tunnel():
    proc = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    for line in proc.stdout:
        decoded = line.decode(errors="replace").strip()
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', decoded)
        if match:
            print(f"\n✅ HF_ENDPOINT_URL={match.group()}\n")
            print("👉 Paste into .env.local → restart npm run dev")
            break

threading.Thread(target=start_tunnel, daemon=True).start()
print("Waiting for tunnel URL…")


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8000
 * Running on http://172.28.0.12:8000
INFO:werkzeug:Press CTRL+C to quit


Waiting for tunnel URL…


In [4]:
import subprocess, re, time

# Check if there's already a cloudflared process running
import os
os.system("pkill -f cloudflared 2>/dev/null; true")
time.sleep(1)

proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)

print("Reading cloudflared output...")
start = time.time()
while time.time() - start < 60:
    line = proc.stdout.readline()
    if line:
        decoded = line.decode(errors="replace").strip()
        print(decoded)
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', decoded)
        if match:
            print(f"\n✅ HF_ENDPOINT_URL={match.group()}")
            break
    else:
        time.sleep(0.1)


Reading cloudflared output...
2026-03-13T06:38:34Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-03-13T06:38:34Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-13T06:38:37Z INF +--------------------------------------------------------------------------------------------+
2026-03-13T06:38:37Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-03-13T06:38:37Z INF |  https://defendant-athle

.









Below is alternative way

In [ ]:
"""import os

hf_token = os.environ.get('HF_TOKEN')
if hf_token:
    print(f"HF_TOKEN is loaded. First few characters: {hf_token[:5]}...")
else:
    print("HF_TOKEN is NOT loaded.")

HF_TOKEN is NOT loaded.


In [ ]:
"""# Load model directly
from transformers import AutoProcessor, AutoModelForImageTextToText
from huggingface_hub import get_token
import os

# Get token from Hugging Face login or environment variable
hf_token = get_token()
if hf_token is None:
    hf_token = os.environ.get('HF_TOKEN')

if hf_token:
    print(f"HF_TOKEN is loaded. First few characters: {hf_token[:5]}...")
else:
    print("HF_TOKEN is NOT loaded.")
    print("Please ensure you have logged in to Hugging Face or set HF_TOKEN as a Colab Secret.")

processor = AutoProcessor.from_pretrained("google/medgemma-1.5-4b-it", token=hf_token)
model = AutoModelForImageTextToText.from_pretrained("google/medgemma-1.5-4b-it", token=hf_token)
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
inputs = processor.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:]))